# 🏥 MedGemma Clinical Triage Assistant
### Kaggle MedGemma Impact Challenge 2026

⚠️ **DISCLAIMER:** Educational purposes only. NOT medical advice.

## Step 1: Setup

In [ ]:
!pip install transformers accelerate torch pillow requests huggingface_hub -q
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Logged in!")

## Step 2: Load MedGemma (CPU Mode - Memory Efficient)

**Note:** Using CPU to avoid GPU OOM errors. Inference takes ~10-15 seconds per case.

In [ ]:
import os
# Set environment variable to prevent memory fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

from transformers import AutoModelForImageTextToText, AutoProcessor, GenerationConfig
import torch

# Force CPU to avoid GPU memory issues
device = "cpu"
print(f"🔄 Device: {device}")
print("⏳ Loading MedGemma... this may take 1-2 minutes...")

# Load model with memory-efficient settings
model = AutoModelForImageTextToText.from_pretrained(
    "google/medgemma-1.5-4b-it",
    dtype=torch.float32,  # CPU-compatible (using dtype instead of deprecated torch_dtype)
    low_cpu_mem_usage=True,
    device_map="cpu"
)

processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it", trust_remote_code=True)

# Set generation config directly on model
model.generation_config = GenerationConfig(
    max_new_tokens=256,  # Increased for complete responses
    do_sample=False,
    pad_token_id=1
)

# Create pipeline WITHOUT generation_config parameter
from transformers import pipeline
pipe = pipeline(
    "image-text-to-text",
    model=model,
    processor=processor,
    device=device
)

print("✅ MedGemma-1.5-4b-it loaded successfully!")

## Step 3: Triage Function

In [ ]:
def triage(symptoms, age, gender, history=""):
    # Shorter prompt for faster inference
    prompt = f"""Clinical triage for {age}yo {gender}:
Symptoms: {symptoms}
History: {history or 'None'}

Triage level (LOW/MEDIUM/HIGH): """
    
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    output = pipe(text=messages, max_new_tokens=256)
    return output[0]["generated_text"][-1]["content"]

def show(name, patient, result):
    print(f"\n{'='*60}\n📋 {name}\n{'='*60}")
    print(f"Patient: {patient['age']}yo {patient['gender']}")
    print(f"Symptoms: {patient['symptoms']}")
    print(f"\n🤖 MedGemma:\n{result}\n{'='*60}")

print("✅ Ready!")

## Step 4: Demo Cases

In [ ]:
# Case 1: HIGH
print("🚨 Case 1: Chest Pain")
case1 = {"age": 55, "gender": "Male", "symptoms": "Crushing chest pain, shortness of breath, cold sweat", "history": "Hypertension, smoker"}
r1 = triage(**case1)
show("Chest Pain (HIGH)", case1, r1)

In [ ]:
# Case 2: MEDIUM
print("\n🟡 Case 2: Abdominal Pain")
case2 = {"age": 25, "gender": "Female", "symptoms": "Right lower quadrant pain, nausea, fever", "history": "None"}
r2 = triage(**case2)
show("Abdominal Pain (MEDIUM)", case2, r2)

In [ ]:
# Case 3: LOW
print("\n🟢 Case 3: Mild Headache")
case3 = {"age": 32, "gender": "Male", "symptoms": "Mild tension headache, no other symptoms", "history": "Occasional headaches"}
r3 = triage(**case3)
show("Mild Headache (LOW)", case3, r3)

## Conclusion

✅ MedGemma-1.5-4b-it working  
✅ Clinical triage functional  
✅ Edge AI ready (CPU mode)  

**MedGemma Impact Challenge 2026 | CC BY 4.0**